# Scaling computations with parallel computing

In [1]:
println("Number of threads that your Julia is run: ## $(Threads.nthreads())")

Number of threads that your Julia is run: ## 8


In [2]:
using BenchmarkTools, Distributed

## Multithreading

Make a 1e4 element vector with random numbers between 0 and 1e3 using multithreading.

In [3]:
function tsum(x)
    r, c = size(x)
    y = zeros(c)
    Threads.@threads for i in 1:c
        for j in 1:r
            @inbounds y[i] += x[j, i]
        end
    end
    y
end

x = rand(1000,10000);
@time tsum(x)

  0.066088 seconds (50.91 k allocations: 3.560 MiB, 328.57% compilation time)


10000-element Vector{Float64}:
 507.32063850115264
 489.27440290980553
 496.6910388467916
 488.96946576198866
 501.4887486219914
 491.45460504672525
 499.18476257680766
 507.0435053923319
 507.64092584604134
 504.79698959880324
 506.1871641171471
 498.84624456277805
 485.99980620719685
   ⋮
 491.1762864598846
 483.39735170430805
 499.455074236958
 496.9191646140306
 508.74337635127404
 505.3286313940505
 497.21062105365166
 522.6612010468041
 504.81281915280897
 505.22781266301064
 508.62835286251425
 507.0712892096479

### Locking mechanism for threads
Atomic, spin, reentrant. 

## Green threading
Handles threads in the user-space rather than kernel-space.

In [4]:
@time sleep(2)

  2.001018 seconds (45 allocations: 1.031 KiB)


In [5]:
@time t = @async sleep(4)

  0.006659 seconds (2.80 k allocations: 201.828 KiB, 99.02% compilation time)


Task (runnable) @0x0000741c9724bd00

Run some tasks asynchronously.

In [6]:
function dojob(i)
    val = round(rand(), digits=2)
    sleep(val)
    i, val
end

result = Vector{Tuple{Int,Float64}}(undef, 8);
@time @sync for i=1:8
    @async result[i] = dojob(i)
end
result

  0.954027 seconds (11.64 k allocations: 704.562 KiB, 2.83% compilation time)


8-element Vector{Tuple{Int64, Float64}}:
 (1, 0.6)
 (2, 0.22)
 (3, 0.06)
 (4, 0.93)
 (5, 0.08)
 (6, 0.68)
 (7, 0.12)
 (8, 0.36)

## Multi-processing and distributed computing
Use 4 workers.

In [7]:
addprocs(max(0, 5-nprocs()));
workers()

4-element Vector{Int64}:
 2
 3
 4
 5

In [8]:
function p_rand()
    n = 10^4
    x = @distributed (+) for i in 1:n
        # the last line will be aggregated
        sum(rand(10^4))
    end
    x / n
end
@time p_rand()

  1.135337 seconds (486.27 k allocations: 32.676 MiB, 35.33% compilation time)


4999.6851878230245

In [9]:
workers()'

1×4 adjoint(::Vector{Int64}) with eltype Int64:
 2  3  4  5

In [10]:
function myf()
    println("I am on worker ", myid())
    rand()
end
myf()

I am on worker 1


0.026460924043368084

Try to run on worker 4 only.

In [11]:
try
    fetch(@spawnat 4 myf())
catch e
    println(e)
end

RemoteException(4, CapturedException(UndefVarError(Symbol("#myf")), Any[(deserialize_datatype at Serialization.jl:1399, 1), (handle_deserialize at Serialization.jl:867, 1), (deserialize at Serialization.jl:814, 1), (handle_deserialize at Serialization.jl:874, 1), (deserialize at Serialization.jl:814 [inlined], 1), (deserialize_global_from_main at clusterserialize.jl:160, 1), (#5 at clusterserialize.jl:72 [inlined], 1), (foreach at abstractarray.jl:3094, 1), (deserialize at clusterserialize.jl:72, 1), (handle_deserialize at Serialization.jl:960, 1), (deserialize at Serialization.jl:814, 1), (handle_deserialize at Serialization.jl:871, 1), (deserialize at Serialization.jl:814, 1), (handle_deserialize at Serialization.jl:874, 1), (deserialize at Serialization.jl:814 [inlined], 1), (deserialize_msg at messages.jl:87, 1), (#invokelatest#2 at essentials.jl:887 [inlined], 1), (invokelatest at essentials.jl:884 [inlined], 1), (message_handler_loop at process_messages.jl:176, 1), (process_tcp_s

Need to use `@everywhere`.

In [12]:
@everywhere function myf()
    println("I'm on worker ", myid())
    rand()
end
fetch(@spawnat 4 myf())

      From worker 4:	I'm on worker 4


0.5197978300580908

### A typical pattern for setting an initial state across workers

In [14]:
using Distributed
@everywhere using Pkg
@everywhere Pkg.activate("../")
@everywhere using Distributed, Random, DataFrames

@everywhere function calc(x, y)
    2x + y
end

@everywhere function init_worker()    
   Random.seed!(myid())
    # reading initial data from files or other actions
end

@sync for wid in workers()
    @async fetch(@spawnat wid init_worker())
end

  Activating project at `~/code/julia-learning`


      From worker 3:	  Activating project at `~/code/julia-learning`
      From worker 2:	  Activating project at `~/code/julia-learning`
      From worker 4:	  Activating project at `~/code/julia-learning`
      From worker 5:	  Activating project at `~/code/julia-learning`


Perform a distributed calculation.

In [15]:
data = @distributed (append!) for (i, j) = vec(collect(Iterators.product(1:4, 1:3)))
    a = rand(1:499)
    b = rand(1:9)*1000
    c = calc(a, b)
    DataFrame(;i,j,a,b,c,procid = myid())
end

Row,i,j,a,b,c,procid
,Int64,Int64,Int64,Int64,Int64,Int64
1,1,1,158,9000,9316,2
2,2,1,17,8000,8034,2
3,3,1,428,1000,1856,2
4,4,1,197,1000,1394,3
5,1,2,399,5000,5798,3
6,2,2,123,4000,4246,3
7,3,2,207,3000,3414,4
8,4,2,499,4000,4998,4
9,1,3,285,7000,7570,4


Single Instruction Multiple Data (SIMD): `@simd' can parallelise in this case.